# Neural Network Training: Activation Functions and Optimisation

**Dataset:** H2 molecule potential energy surface (FCI reference, 451 points)

---

## The Challenge

Quantum chemistry calculations are **extremely expensive** — computing the energy of H2 at just
one geometry can take minutes of CPU time. But to run molecular dynamics or scan a reaction
pathway, we need energies at *thousands* of geometries.

**The solution:** Train a neural network on a small number of quantum calculations, then use
the trained network to predict the energy at any new geometry almost instantly.

```
  H2 molecule:     H ---- H
                   R (Ang) = internuclear distance

  FCI calculation: ~minutes per point (expensive)
  Trained NN:      ~microseconds per point (fast)
```

**Our goal:** Train a neural network on only **20 data points** and have it predict the
entire potential energy curve accurately.

---

## Lab Roadmap

| Part | Topic |
|------|-------|
| 1-5 | Setup, data, architectures, training (locked, run only) |
| **3** | Exercise: Visualise the dataset |
| **4** | Exercise: Understand activation functions |
| **5** | Exercise: Implement a training step |
| **7** | Exercise: Plot training dynamics |
| **8** | Exercise: Visualise predictions |
| **9** | Exercise: Compute evaluation metrics |

---

## Glossary

| Term |  |
|------|---------------|
| **Hartree** | Unit of energy in quantum chemistry (1 Ha = 27.2 eV = 627 kcal/mol) |
| **FCI** | Full Configuration Interaction -- the most accurate quantum chemistry method |
| **Potential Energy Surface (PES)** | How energy changes as atoms move |
| **Activation function** | A mathematical function applied after each NN layer to add non-linearity |
| **RBF (Radial Basis Function)** | A Gaussian-shaped activation: exp(-a(x-c)^2) |
| **Epoch** | One full pass through all training data |
| **Gradient descent** | Updating weights by moving 'downhill' on the loss surface |
| **Overfitting** | Model memorises training data but fails on new data |
| **Validation set** | Data held out during training to monitor generalisation |
| **MAE** | Mean Absolute Error -- average size of prediction errors |
| **Chemical accuracy** | 1 kcal/mol = 0.0016 Hartree -- the accuracy target for chemistry |

---

## Quick Python / PyTorch Reference

```python
import torch
import torch.nn as nn

x = torch.tensor([[1.0], [2.0]])  # shape (2, 1)
layer = nn.Linear(1, 4)           # 1 input -> 4 outputs
out   = layer(x)                  # shape (2, 4)

# Loss and backward pass
loss_fn = nn.MSELoss()
loss    = loss_fn(predictions, targets)
loss.backward()                   # compute gradients
optimizer.step()                  # update weights

# Turn off gradients (for eval / testing)
with torch.no_grad():
    pred = model(x_test)

# Convert tensor to numpy
arr = tensor.numpy()
```


---
# Part 1: Setup

> **Just run these cells — no code to fill in.**

In [ ]:
# Install/upgrade required packages
!pip install --upgrade matplotlib torch -q
print("Packages installed successfully!")

In [ ]:
# # Download H2 molecule data
# !wget -q https://raw.githubusercontent.com/dralgroup/MLinQCbook22-NN/main/R_451.dat
# !wget -q https://raw.githubusercontent.com/dralgroup/MLinQCbook22-NN/main/E_FCI_451.dat
# print("Dataset downloaded!")
# print("- R_451.dat: Internuclear distances (451 points)")
# print("- E_FCI_451.dat: FCI energies (451 points)")

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.utils.data as data
import matplotlib.pyplot as plt
from copy import deepcopy

# Set random seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)

# Configure plotting
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

print(f"PyTorch version: {torch.__version__}")
print(f"NumPy version: {np.__version__}")

---
# Part 2: Data — The H₂ Potential Energy Surface 

The H₂ molecule is the simplest possible molecule: two hydrogen atoms.
Its potential energy depends only on one number: the distance R between the two nuclei.

```
         Short R: atoms repel   Equilibrium: minimum energy   Long R: atoms separate
          ↑ E                       ↓ E                            → 0
         H·H                      H─H                             H  H
```

We have **451 FCI reference points** spanning R = 0.5 to 5.0 Å.  
We'll train on only **20 evenly-spaced points** — a very sparse sample!

> **Just run these cells.**


In [ ]:
# Load all data points
X_all = np.loadtxt('R_451.dat', dtype=np.float32)
Y_all = np.loadtxt('E_FCI_451.dat', dtype=np.float32)

print(f"Total data points: {len(X_all)}")
print(f"Distance range: [{X_all.min():.2f}, {X_all.max():.2f}] Å")
print(f"Energy range: [{Y_all.min():.4f}, {Y_all.max():.4f}] Hartree")

# Sparse sampling: Select only 20 points for training
train_num = 20
val_ratio = 0.2  # 20% of training set for validation
val_num = int(train_num * val_ratio)

# Select evenly spaced training points
train_idx = np.linspace(0, len(X_all)-1, train_num, dtype=np.int32)
X_train = X_all[train_idx].reshape(-1, 1)
Y_train = Y_all[train_idx].reshape(-1, 1)

# Remaining points are test set
test_idx = np.setdiff1d(np.arange(len(X_all)), train_idx)
X_test = X_all[test_idx].reshape(-1, 1)
Y_test = Y_all[test_idx].reshape(-1, 1)

print(f"\nData split:")
print(f"- Training: {len(X_train)} points")
print(f"- Testing: {len(X_test)} points")

In [ ]:
# Further split training into train and validation
# Stratified split: pick val points evenly spaced across the sorted training set
# so the validation set covers the full energy range (well minimum + plateau).
# Random splitting with seed 42 accidentally clusters all val points in the
# plateau region, giving a poor signal for early stopping.
np.random.seed(1)
val_idx = np.linspace(0, train_num - 1, val_num, dtype=int)  # evenly spaced
train_idx_final = np.setdiff1d(np.arange(train_num), val_idx)

# Convert to PyTorch tensors
x_train = torch.from_numpy(X_train[train_idx_final])
x_val   = torch.from_numpy(X_train[val_idx])
y_train = torch.from_numpy(Y_train[train_idx_final])
y_val   = torch.from_numpy(Y_train[val_idx])
x_test  = torch.from_numpy(X_test)
y_test  = torch.from_numpy(Y_test)

# Create PyTorch dataset
train_ds = data.TensorDataset(x_train, y_train)

print(f'Final split:')
print(f'- Training:   {len(x_train)} points  (R range: [{x_train.numpy().min():.2f}, {x_train.numpy().max():.2f}] Ang)')
print(f'- Validation: {len(x_val)} points  (R values: {[f"{v:.2f}" for v in x_val.numpy().flatten()]})')
print(f'- Testing:    {len(x_test)} points')
print()
print('Validation points cover: well region, shoulder, mid-range, and plateau.')


---
# Part 3: Visualise the Dataset 

Before training any model, always **look at your data**.

---

###  Exercise 3.1 — Plot the H₂ Potential Energy Curve (5 points)

Create a single figure showing:
- The full reference FCI curve (blue line, 451 points)  
- Training points (red circles)
- Validation points (green circles)

Label axes, add a title and legend.

<details>
<summary>💡 Hint: matplotlib scatter and plot (click to expand)</summary>

```python
fig, ax = plt.subplots(figsize=(10, 6))

# Reference curve (full 451 points)
ax.plot(X_all, Y_all, 'b-', linewidth=2, label='Reference FCI (451 pts)', alpha=0.7)

# Training points — note: .numpy() converts a tensor to a numpy array
ax.scatter(x_train.numpy(), y_train.numpy(),
           c='red', s=80, edgecolors='black', zorder=5,
           label=f'Training ({len(x_train)} pts)')

# Validation points
ax.scatter(x_val.numpy(), y_val.numpy(),
           c='green', s=80, edgecolors='black', zorder=5,
           label=f'Validation ({len(x_val)} pts)')

ax.set_xlabel(r'Internuclear Distance R ($\AA$)')
ax.set_ylabel('Energy (Hartree)')
ax.set_title('...')
ax.legend()
ax.grid(alpha=0.3)
plt.show()
```
</details>


In [ ]:
# YOUR CODE HERE

fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(X_all, Y_all, 'b-', linewidth=2,
        label=f'Reference FCI (451 pts)', alpha=0.7)
ax.scatter(x_train.numpy(), y_train.numpy(),
           c='red', s=80, edgecolors='black', linewidth=1.5, zorder=5,
           label=f'Training ({len(x_train)} pts)')
ax.scatter(x_val.numpy(), y_val.numpy(),
           c='green', s=80, edgecolors='black', linewidth=1.5, zorder=5,
           label=f'Validation ({len(x_val)} pts)')

ax.set_xlabel(r'Internuclear Distance R ($\AA$)', fontsize=12)
ax.set_ylabel('Energy (Hartree)', fontsize=12)
ax.set_title('H₂ Potential Energy Curve — Training Data', fontsize=14)
ax.legend(fontsize=10)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Challenge: learn the smooth curve from only {len(x_train)} training points!")


In [ ]:
# Tests
assert len(X_all) == 451, "X_all should have 451 points"
assert len(x_train) + len(x_val) == 20, "train + val should sum to 20"
assert len(x_test) > 400, "Test set should have ~431 points"
assert x_train.shape[1] == 1, "x_train should be shape (n, 1)"
assert y_train.shape[1] == 1, "y_train should be shape (n, 1)"

# Check that val set is well distributed (covers both well and plateau)
val_r = x_val.numpy().flatten()
assert val_r.min() < 1.5, \
    f"Val set should include the well region (R < 1.5 Ang), min val R = {val_r.min():.2f}"
assert val_r.max() > 3.0, \
    f"Val set should include the plateau region (R > 3.0 Ang), max val R = {val_r.max():.2f}"

print('Dataset correctly set up!')
print(f'   {len(x_train)} train + {len(x_val)} val + {len(x_test)} test')
print(f'   Val R values: {[f"{v:.2f}" for v in sorted(val_r)]} Ang')
print(f'   Val covers well (R < 1.5 Ang): yes')
print(f'   Val covers plateau (R > 3.0 Ang): yes')


---
# Part 4: Neural Network Architectures 

## Why activation functions matter

Without an activation function, stacking multiple linear layers still gives you just a linear model:
```
  Layer 1: y = W₁x + b₁
  Layer 2: y = W₂(W₁x + b₁) + b₂  =  (W₂W₁)x + (W₂b₁ + b₂)
                                    =  a single linear function!
```

An activation function **breaks this linearity** so the network can learn curves, not just lines.

## The Radial Basis Function (RBF)

$$\text{RBF}(x) = \exp\!\left(-a\,(x - c)^2\right)$$

This is a **Gaussian bump**: it's 1 at centre c, falls off symmetrically, and the width is
controlled by a. Both c and a are **learned parameters** — the network adjusts them during training.

> **Run the two cells below (RBF class + architectures) — no code to fill in.**


In [ ]:
class RBF(nn.Module):
    """Radial Basis Function activation."""
    def __init__(self, in_feature: int):
        super().__init__()
        # Learnable parameters
        self.c = nn.Parameter(torch.Tensor(in_feature))  # Centers
        self.a = nn.Parameter(torch.Tensor(1))  # Width parameter
        
        # Initialize parameters
        nn.init.normal_(self.c, mean=3.0, std=1.0)
        nn.init.normal_(self.a, mean=1.0, std=0.5)
    
    def forward(self, input):
        # Expand centers to match batch size
        c = self.c.unsqueeze(0).expand(input.size(0), -1)
        # Compute Gaussian RBF
        return torch.exp(-self.a * (input - c).pow(2))

print("RBF activation function defined!")
print("Formula: RBF(x) = exp(-a * (x - c)²)")

In [ ]:
class NN_RBF(nn.Module):
    """Neural Network with RBF activation functions."""
    def __init__(self, *features):
        super().__init__()
        layers = []
        for i in range(len(features) - 1):
            layers.append(nn.Linear(features[i], features[i+1]))
            if i < len(features) - 2:  # Don't add activation after last layer
                layers.append(RBF(features[i+1]))
        self.model = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.model(x)

class NN_Linear(nn.Module):
    """Neural Network with no activation (linear only)."""
    def __init__(self, *features):
        super().__init__()
        layers = []
        for i in range(len(features) - 1):
            layers.append(nn.Linear(features[i], features[i+1]))
        self.model = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.model(x)

print("Network architectures defined:")
print("- NN_Linear: Linear layers only (no activation)")
print("- NN_RBF: Linear layers + RBF activations")

---
### Exercise 4.1 — Visualise Activation Functions 

Understanding what an activation function *looks like* helps you understand what the
network can learn.

**Your task:** Create a 2-panel figure comparing:

**Panel a)** The RBF activation with different parameter values — show how c (centre)
and a (width) control the shape. Plot at least 3 different settings.

**Panel b)** RBF vs. "linear activation" (no activation = identity function y=x)  
on the same axes to make the comparison clear.

<details>
<summary>💡 Hint: how to evaluate an RBF layer (click to expand)</summary>

```python
x_demo = torch.linspace(-2, 8, 200).unsqueeze(1)  # shape (200, 1)

# Create an RBF layer and set its parameters manually
rbf_layer = RBF(in_feature=1)
rbf_layer.c.data = torch.tensor([3.0])   # centre at 3.0
rbf_layer.a.data = torch.tensor([1.0])   # width = 1.0

with torch.no_grad():
    y_rbf = rbf_layer(x_demo)            # shape (200, 1)
    
ax.plot(x_demo.numpy(), y_rbf.numpy(), label='RBF c=3.0, a=1.0')
```
</details>

<details>
<summary>💡 Hint: how to show "no activation" (click to expand)</summary>

The "linear" model has no activation at all — you can represent that as:
```python
ax.plot(x_demo.numpy(), x_demo.numpy(), 'k--', label='No activation (y = x)')
```
</details>


In [ ]:
# YOUR CODE HERE

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
x_demo = torch.linspace(0, 6, 300).unsqueeze(1)

# ── Panel a): RBF with different parameters ──────────────────────────
ax = axes[0]
configs = [
    (3.0, 0.5,  'tomato',     'c=3.0, a=0.5  (wide)'),
    (3.0, 2.0,  'steelblue',  'c=3.0, a=2.0  (narrow)'),
    (1.5, 1.0,  'forestgreen','c=1.5, a=1.0  (shifted left)'),
    (4.5, 1.0,  'purple',     'c=4.5, a=1.0  (shifted right)'),
]
for c_val, a_val, color, label in configs:
    rbf = RBF(in_feature=1)
    rbf.c.data = torch.tensor([c_val])
    rbf.a.data = torch.tensor([a_val])
    with torch.no_grad():
        y = rbf(x_demo)
    ax.plot(x_demo.numpy(), y.numpy(), color=color, lw=2, label=label)

ax.set_xlabel('Input x', fontsize=12)
ax.set_ylabel('RBF(x) = exp(−a(x−c)²)', fontsize=12)
ax.set_title('RBF Activation: Effect of Parameters c and a', fontsize=12)
ax.legend(fontsize=9)
ax.grid(alpha=0.3)
ax.text(0.02, 0.97, 'a)', transform=ax.transAxes, fontsize=14, fontweight='bold', va='top')

# ── Panel b): RBF vs. no activation ──────────────────────────────────
ax2 = axes[1]
rbf_std = RBF(in_feature=1)
rbf_std.c.data = torch.tensor([3.0])
rbf_std.a.data = torch.tensor([1.0])
with torch.no_grad():
    y_rbf = rbf_std(x_demo)

ax2.plot(x_demo.numpy(), y_rbf.numpy(), 'steelblue', lw=2.5, label='RBF activation (c=3, a=1)')
ax2.plot(x_demo.numpy(), x_demo.numpy(), 'k--', lw=2, alpha=0.7, label='No activation (y = x)')
ax2.axhline(0, color='gray', lw=0.8, alpha=0.5)
ax2.set_xlabel('Input x', fontsize=12)
ax2.set_ylabel('Output', fontsize=12)
ax2.set_title('RBF vs. No Activation\n(non-linearity enables curve fitting)', fontsize=12)
ax2.legend(fontsize=10)
ax2.grid(alpha=0.3)
ax2.text(0.02, 0.97, 'b)', transform=ax2.transAxes, fontsize=14, fontweight='bold', va='top')

plt.tight_layout()
plt.show()

print("Key observations:")
print("  • RBF is bounded [0, 1] — the network can create 'bumps' and 'dips'")
print("  • Centre c shifts the bump horizontally — network can focus on any R")
print("  • Width a controls sharpness — learned automatically during training")
print("  • Linear (no activation) is just a straight line — can't fit a curve!")


In [ ]:
# Tests
# Verify RBF layer behaves correctly
import torch
rbf_test = RBF(in_feature=1)
rbf_test.c.data = torch.tensor([3.0])
rbf_test.a.data = torch.tensor([1.0])
x_test_rbf = torch.tensor([[3.0]])  # At center, should output 1.0
with torch.no_grad():
    out_at_center = rbf_test(x_test_rbf).item()
assert abs(out_at_center - 1.0) < 1e-5, \
    f"RBF at center should be 1.0, got {out_at_center:.6f}"

x_far = torch.tensor([[10.0]])  # Far from center, should be near 0
with torch.no_grad():
    out_far = rbf_test(x_far).item()
assert out_far < 0.01, f"RBF far from center should be near 0, got {out_far:.6f}"

print("RBF activation behaves correctly.")
print(f"   RBF(c=3, a=1) at x=3.0: {out_at_center:.6f}  (expected 1.0)")
print(f"   RBF(c=3, a=1) at x=10: {out_far:.8f}  (expected ~0)")


---
# Part 5: The Training Loop 

## How neural networks learn

Training a neural network is like adjusting the dials on a machine until it
produces the right output. Each "dial" is a weight or bias parameter.

```
  ┌──────────────────────────────────────────────────────────┐
  │  Repeat for each epoch:                                  │
  │                                                          │
  │  1. FORWARD PASS: feed input → get prediction            │
  │     pred = model(x)                                      │
  │                                                          │
  │  2. COMPUTE LOSS: how wrong are we?                      │
  │     loss = MSE(pred, true_energy)                        │
  │                                                          │
  │  3. BACKWARD PASS: which parameters caused the error?    │
  │     optimizer.zero_grad()   ← clear old gradients        │
  │     loss.backward()         ← compute new gradients      │
  │                                                          │
  │  4. UPDATE WEIGHTS: move downhill on loss surface        │
  │     optimizer.step()                                     │
  │                                                          │
  │  5. VALIDATION: check on held-out data                   │
  │     save model if best so far                            │
  └──────────────────────────────────────────────────────────┘
```

> **The full training function below is provided — just run it.**


In [ ]:
def train(model, dataset, x_val, y_val, batch_size, epoch_num=10000, print_every=1000):
    """
    Train a neural network model.

    Args:
        model      : Neural network to train
        dataset    : Training TensorDataset
        x_val, y_val : Validation tensors
        batch_size : Batch size
        epoch_num  : Number of training epochs
        print_every: Print progress every this many epochs

    Returns:
        best_model     : Model with lowest validation loss (built-in early stopping)
        train_loss_arr : Training MSE per epoch  (numpy array, length epoch_num)
        val_loss_arr   : Validation MSE per epoch (numpy array, length epoch_num)
    """
    loss_fn   = nn.MSELoss()
    optimizer = optim.Adam(params=model.parameters())

    best_loss  = float('inf')
    best_model = None
    best_epoch = 0
    train_loss_list = []
    val_loss_list   = []

    model.train()
    for epoch_idx in range(1, epoch_num + 1):
        train_loss = []
        for (batch_x, batch_y) in data.DataLoader(dataset, batch_size=batch_size, shuffle=True):
            pred = model(batch_x)
            loss = loss_fn(pred, batch_y)
            model.zero_grad()   # clear gradients inside the loop (one clear per batch)
            loss.backward()
            optimizer.step()
            train_loss.append(loss.detach().item())
        train_loss_list.append(np.mean(train_loss))

        model.eval()
        with torch.no_grad():
            val_loss = loss_fn(model(x_val), y_val).item()
        val_loss_list.append(val_loss)
        if val_loss < best_loss:
            best_loss  = val_loss
            best_epoch = epoch_idx
            best_model = deepcopy(model)
        model.train()

        if epoch_idx % print_every == 0:
            print(f'Epoch {epoch_idx:5d}: Train Loss = {train_loss_list[-1]:.4e}, '
                  f'Val Loss = {val_loss:.4e}')

    print(f'Best validation loss: {best_loss:.4e}  (epoch {best_epoch})')
    return best_model, np.array(train_loss_list), np.array(val_loss_list)

print('Training function ready!')


---
###  Exercise 5.1 — Implement a Single Training Step

The locked `train()` function handles the full loop. Here you implement the
**core inner step** from scratch to understand what's happening inside.

Complete `training_step` which:
1. Computes the model's predictions for input `x`
2. Computes the MSE loss against the true values `y`
3. Clears old gradients
4. Runs the backward pass
5. Updates the model weights

Then run it for 500 steps and watch the loss decrease.

<details>
<summary>💡 Step-by-step hint (click to expand)</summary>

```python
def training_step(model, optimizer, loss_fn, x, y):
    # 1. Forward pass: get predictions
    pred = model(x)
    
    # 2. Compute loss
    loss = loss_fn(pred, y)
    
    # 3. Clear old gradients (always do this before backward!)
    optimizer.zero_grad()
    
    # 4. Backward pass: compute gradients
    loss.backward()
    
    # 5. Update weights
    optimizer.step()
    
    return loss.item()   # return the loss value as a Python float
```
</details>


In [ ]:



import torch.optim as optim

def training_step(model, optimizer, loss_fn, x, y):
    """
    Perform a single gradient descent step.
    
    Parameters
    ----------
    model     : nn.Module   — the neural network
    optimizer : Optimizer   — e.g. Adam
    loss_fn   : loss        — e.g. nn.MSELoss()
    x, y      : Tensors     — inputs and targets for this batch
    
    Returns
    -------
    loss_val : float  — the loss value after this step
    """
    # YOUR CODE HERE
    raise NotImplementedError()

# ── Demo: watch a small RBF network learn ────────────────────────────
torch.manual_seed(0)
demo_model = NN_RBF(1, 16, 1)
demo_opt   = optim.Adam(demo_model.parameters(), lr=0.01)
demo_loss  = nn.MSELoss()

losses = []
for step in range(500):
    l = training_step(demo_model, demo_opt, demo_loss, x_train, y_train)
    losses.append(l)

print(f"Training step demo (500 steps on x_train / y_train):")
print(f"  Loss at step   1: {losses[0]:.6f}")
print(f"  Loss at step 100: {losses[99]:.6f}")
print(f"  Loss at step 500: {losses[-1]:.6f}")
print(f"  Total reduction : {losses[0]/losses[-1]:.1f}× " if losses[-1]>0 else "")

fig, ax = plt.subplots(figsize=(8, 4))
ax.semilogy(losses, 'steelblue', lw=2)
ax.set_xlabel('Training Step'); ax.set_ylabel('MSE Loss (log scale)')
ax.set_title('Loss Decreasing Over 500 Training Steps\n(gradient descent working correctly)', fontsize=12)
ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()



In [ ]:
# Tests
# Test that training_step actually reduces the loss
torch.manual_seed(99)
test_model = NN_RBF(1, 8, 1)
test_opt   = optim.Adam(test_model.parameters(), lr=0.01)
test_lossfn = nn.MSELoss()

loss_before = test_lossfn(test_model(x_train), y_train).item()
l1 = training_step(test_model, test_opt, test_lossfn, x_train, y_train)
loss_after  = test_lossfn(test_model(x_train), y_train).item()

assert isinstance(l1, float), "training_step must return a float"
assert l1 > 0, "Loss must be positive"
assert loss_after < loss_before, \
    f"Loss should decrease after a training step! Before={loss_before:.6f}, After={loss_after:.6f}"

# Check gradients are zeroed properly (forgetting zero_grad causes gradient accumulation and divergence)
torch.manual_seed(7)
check_model = NN_RBF(1, 4, 1)
check_opt   = optim.Adam(check_model.parameters(), lr=0.001)
check_loss  = nn.MSELoss()
prev_loss = float('inf')
diverged = False
for _ in range(20):
    l = training_step(check_model, check_opt, check_loss, x_train, y_train)
    if l > prev_loss * 10:
        diverged = True; break
    prev_loss = l
assert not diverged, "Loss diverged -- did you forget optimizer.zero_grad()?"

print("training_step correct!")
print(f"   Loss before step: {loss_before:.6f}")
print(f"   Loss after  step: {loss_after:.6f}  (decreased)")


---
# Part 6: Train Both Models 

Now we use the full `train()` function to train both models for 10,000 epochs each.
This may take 1–2 minutes. Watch the loss values printed below.

> **Just run these cells.**


In [ ]:
print('=' * 60)
print('Training Linear Model (No Activation)')
print('=' * 60)

torch.manual_seed(1)
linear_model = NN_Linear(1, 64, 1)
linear_model, train_loss_linear, val_loss_linear = train(
    linear_model, train_ds, x_val, y_val,
    batch_size=len(train_idx_final),
    epoch_num=10000,
    print_every=1000
)

In [ ]:
print()
print('=' * 60)
print('Training RBF Model (RBF Activation)')
print('=' * 60)

torch.manual_seed(1)
rbf_model = NN_RBF(1, 64, 1)
rbf_model, train_loss_rbf, val_loss_rbf = train(
    rbf_model, train_ds, x_val, y_val,
    batch_size=len(train_idx_final),
    epoch_num=10000,
    print_every=1000
)


---
# Part 7: Analyse Training Dynamics 

---

###  Exercise 7.1 — Plot Training and Validation Losses 

Create **side-by-side plots** of the training dynamics for both models.
Use a **log scale** on the y-axis so you can see the differences clearly.

Each panel should show both the training loss and the validation loss on the same axes.

<details>
<summary>💡 Hint (click to expand)</summary>

```python
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left panel: linear model
axes[0].plot(train_loss_linear, label='Train Loss', lw=2)
axes[0].plot(val_loss_linear,   label='Val Loss',   lw=2)
axes[0].set_yscale('log')          # ← log scale to see small differences
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('MSE Loss (log scale)')
axes[0].set_title('Linear Model')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Right panel: RBF model
# ... (same pattern) ...

plt.tight_layout()
plt.show()
```
</details>


In [ ]:
# YOUR CODE HERE
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

best_linear_epoch = int(val_loss_linear.argmin()) + 1
best_rbf_epoch    = int(val_loss_rbf.argmin()) + 1

for ax, tr_loss, vl_loss, title, best_ep, panel in zip(
    axes,
    [train_loss_linear, train_loss_rbf],
    [val_loss_linear,   val_loss_rbf],
    ['Linear Model (No Activation)', 'RBF Model (RBF Activation)'],
    [best_linear_epoch, best_rbf_epoch],
    ['a)', 'b)']
):
    epochs = range(1, len(tr_loss)+1)
    ax.semilogy(epochs, tr_loss, label='Train Loss',      lw=2, alpha=0.85, color='steelblue')
    ax.semilogy(epochs, vl_loss, label='Validation Loss', lw=2, alpha=0.85, color='darkorange')
    # Mark best validation epoch
    ax.axvline(best_ep, color='green', linestyle='--', alpha=0.7,
               label=f'Best val epoch = {best_ep}')
    ax.set_xlabel('Epoch', fontsize=11)
    ax.set_ylabel('MSE Loss (log scale)', fontsize=11)
    ax.set_title(title, fontsize=12)
    ax.legend(fontsize=10)
    ax.grid(alpha=0.3)
    ax.text(0.02, 0.98, panel, transform=ax.transAxes,
            fontsize=14, fontweight='bold', va='top')

plt.suptitle('Training and Validation Loss Over Time', fontsize=13)
plt.tight_layout()
plt.show()

print('Training Loss Summary:')
print(f'  Linear -- final train: {train_loss_linear[-1]:.4e},  best val: {min(val_loss_linear):.4e}  (epoch {best_linear_epoch})')
print(f'  RBF    -- final train: {train_loss_rbf[-1]:.4e},  best val: {min(val_loss_rbf):.4e}  (epoch {best_rbf_epoch})')
print(f'  Best val improvement: {min(val_loss_linear)/min(val_loss_rbf):.0f}x better with RBF')
print()
print('Observation: RBF val loss rises after its best epoch -- this is overfitting.')
print('The train() function saves the best model at the green dashed line.')


In [ ]:
# Tests
assert len(train_loss_linear) == 10000, "Expected 10000 training loss values"
assert len(train_loss_rbf)    == 10000, "Expected 10000 training loss values"
assert len(val_loss_linear)   == 10000, "Expected 10000 validation loss values"
assert all(v > 0 for v in train_loss_linear), "All losses should be positive"

# Convergence checks: both models must have learned something
# (not stuck at random-init level, which would be ~0.005 for this data)
best_linear_val = min(val_loss_linear)
best_rbf_val    = min(val_loss_rbf)
assert best_linear_val < 5e-3, \
    f"Linear model did not converge: best val = {best_linear_val:.4e}"
assert best_rbf_val < 5e-3, \
    f"RBF model did not converge: best val = {best_rbf_val:.4e}"

# Training loss must decrease (gradient descent is working)
assert train_loss_linear[0] > train_loss_linear[-1], \
    "Linear training loss should decrease over time"
assert train_loss_rbf[0] > train_loss_rbf[-1], \
    "RBF training loss should decrease over time"

print("Training dynamics verified!")
print(f"  Linear -- best val: {best_linear_val:.4e}  (epoch {int(val_loss_linear.argmin())+1})")
print(f"  RBF    -- best val: {best_rbf_val:.4e}  (epoch {int(val_loss_rbf.argmin())+1})")
print()
print("Note: val loss ranking with only 4 points is unreliable.")
print("The meaningful model comparison is on the 431-point test set in Part 9.")


---
# Part 8: Visualise Model Predictions 

---

###  Exercise 8.1 — Plot Predictions vs. Reference 

Create **side-by-side plots** comparing each model's predictions against the full
reference FCI curve. Include the training and validation points for reference.

To get predictions from the trained models:
```python
linear_model.eval()                                    # switch to eval mode
with torch.no_grad():                                  # no gradients needed
    x_tensor = torch.from_numpy(X_all.reshape(-1, 1))
    pred = linear_model(x_tensor).numpy()              # shape (451, 1)
```

<details>
<summary>💡 Full structure hint (click to expand)</summary>

```python
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, model, pred_arr, title, label in zip(
    axes,
    [linear_model, rbf_model],
    [linear_pred,  rbf_pred],
    ['a) Linear Model', 'b) RBF Model'],
    ['a)', 'b)']
):
    ax.plot(X_all, Y_all,   'b-',  lw=2.5, label='Reference FCI', alpha=0.7)
    ax.plot(X_all, pred_arr,'r--', lw=2,   label='NN Prediction')
    ax.scatter(x_train.numpy(), y_train.numpy(), c='red',   s=60, zorder=5, label='Train')
    ax.scatter(x_val.numpy(),   y_val.numpy(),   c='green', s=60, zorder=5, label='Val')
    ax.set_xlabel(r'R ($\AA$)')
    ax.set_ylabel('Energy (Hartree)')
    ax.set_title(title)
    ax.legend()
    ax.grid(alpha=0.3)
    ax.text(0.02, 0.98, label, transform=ax.transAxes, fontsize=14, fontweight='bold', va='top')
```
</details>


In [ ]:

linear_model.eval()
rbf_model.eval()

with torch.no_grad():
    x_all_tensor = torch.from_numpy(X_all.reshape(-1, 1))
    linear_pred  = linear_model(x_all_tensor).numpy()
    rbf_pred     = rbf_model(x_all_tensor).numpy()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, model_pred, title, panel in zip(
    axes,
    [linear_pred, rbf_pred],
    ['Linear Model (No Activation)', 'RBF Model (RBF Activation)'],
    ['a)', 'b)']
):
    ax.plot(X_all, Y_all,       'b-',  lw=2.5, label='Reference FCI', alpha=0.7)
    ax.plot(X_all, model_pred,  'r--', lw=2,   label='NN Prediction')
    ax.scatter(x_train.numpy(), y_train.numpy(),
               c='red',   s=70, edgecolors='black', lw=1, zorder=5, label='Train')
    ax.scatter(x_val.numpy(),   y_val.numpy(),
               c='green', s=70, edgecolors='black', lw=1, zorder=5, label='Val')
    ax.set_xlabel(r'Internuclear Distance R ($\AA$)', fontsize=11)
    ax.set_ylabel('Energy (Hartree)', fontsize=11)
    ax.set_title(title, fontsize=12)
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)
    ax.text(0.02, 0.98, panel, transform=ax.transAxes, fontsize=14, fontweight='bold', va='top')

plt.suptitle('Neural Network Predictions vs. FCI Reference', fontsize=13, y=1.01)
plt.tight_layout()
#plt.savefig('NN_predictions.png', dpi=300, bbox_inches='tight')
plt.show()
#print("Figure saved as 'NN_predictions.png'")


In [ ]:
# Tests
assert 'linear_pred' in dir(), "linear_pred not defined -- did you run the cell?"
assert 'rbf_pred'    in dir(), "rbf_pred not defined"
assert linear_pred.shape == (451, 1), \
    f"linear_pred shape should be (451,1), got {linear_pred.shape}"
assert rbf_pred.shape == (451, 1), \
    f"rbf_pred shape should be (451,1), got {rbf_pred.shape}"
linear_max_err = np.max(np.abs(linear_pred.flatten() - Y_all))
rbf_max_err    = np.max(np.abs(rbf_pred.flatten() - Y_all))
assert rbf_max_err < linear_max_err, \
    "RBF max error should be less than linear max error"
assert rbf_max_err < 0.5, \
    f"RBF max error {rbf_max_err:.4f} is too large -- check model is in eval mode with torch.no_grad()"
print("Predictions look correct!")
print(f"   Linear max error: {linear_max_err:.4f} Hartree")
print(f"   RBF    max error: {rbf_max_err:.4f} Hartree")


---
# Part 9: Quantitative Evaluation 

---

###  Exercise 9.1 — Calculate Test Set Metrics 

Compute the following on the **test set** for both models:

| Metric | Formula | Meaning |
|--------|---------|---------|
| **MAE** | mean(|pred − true|) | Average error size |
| **MSE** | mean((pred − true)²) | Punishes large errors more |
| **RMSE** | √MSE | Same units as the data |

Then convert MAE to **kcal/mol** (1 Hartree = 627.509 kcal/mol) and check if either
model reaches **chemical accuracy** (< 1 kcal/mol ≈ 0.0016 Hartree).

Finally, create a **bar chart** comparing the MAE of both models.

<details>
<summary>💡 Hint: computing metrics with torch.no_grad (click to expand)</summary>

```python
with torch.no_grad():
    linear_test_pred = linear_model(x_test)
    linear_mae = torch.mean(torch.abs(linear_test_pred - y_test)).item()
    linear_mse = torch.mean((linear_test_pred - y_test)**2).item()
    
    rbf_test_pred = rbf_model(x_test)
    rbf_mae = ...
    rbf_mse = ...
```
</details>

<details>
<summary>💡 Hint: chemical accuracy conversion (click to expand)</summary>

```python
HARTREE_TO_KCAL = 627.509

linear_mae_kcal = linear_mae * HARTREE_TO_KCAL
rbf_mae_kcal    = rbf_mae    * HARTREE_TO_KCAL

print(f"Chemical accuracy threshold: 1 kcal/mol = {1/HARTREE_TO_KCAL:.4f} Hartree")
print(f"Linear MAE: {linear_mae_kcal:.2f} kcal/mol  → {'chemical accuracy reached' if linear_mae_kcal < 1 else 'not yet at chemical accuracy'}")
```
</details>


In [ ]:
# YOUR CODE HERE

HARTREE_TO_KCAL = 627.509

linear_model.eval()
rbf_model.eval()

with torch.no_grad():
    linear_test_pred = linear_model(x_test)
    linear_mae  = torch.mean(torch.abs(linear_test_pred - y_test)).item()
    linear_mse  = torch.mean((linear_test_pred - y_test)**2).item()

    rbf_test_pred = rbf_model(x_test)
    rbf_mae  = torch.mean(torch.abs(rbf_test_pred - y_test)).item()
    rbf_mse  = torch.mean((rbf_test_pred - y_test)**2).item()

linear_rmse = np.sqrt(linear_mse)
rbf_rmse    = np.sqrt(rbf_mse)

# ── Table ─────────────────────────────────────────────────────────────
print("=" * 72)
print(f"{'Test Set Performance Comparison':^72}")
print("=" * 72)
print(f"{'Model':<22} {'MAE (Ha)':>12} {'MSE':>14} {'RMSE (Ha)':>12} {'MAE (kcal/mol)':>16}")
print("-" * 72)
print(f"{'Linear (No Act.)':<22} {linear_mae:>12.6f} {linear_mse:>14.4e} {linear_rmse:>12.6f} {linear_mae*HARTREE_TO_KCAL:>16.2f}")
print(f"{'RBF':<22} {rbf_mae:>12.6f} {rbf_mse:>14.4e} {rbf_rmse:>12.6f} {rbf_mae*HARTREE_TO_KCAL:>16.2f}")
print("-" * 72)
print(f"{'Improvement (RBF/Lin)':<22} {linear_mae/rbf_mae:>12.1f}×")
print("=" * 72)

chem_acc = 1 / HARTREE_TO_KCAL
print(f"\nChemical accuracy threshold: {chem_acc:.4f} Ha = 1.00 kcal/mol")
print(f"Linear model: {'reaches chemical accuracy' if linear_mae < chem_acc else 'does NOT reach chemical accuracy'}")
print(f"RBF model   : {'reaches chemical accuracy' if rbf_mae    < chem_acc else 'does NOT reach chemical accuracy'}")

# ── Bar chart ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

models  = ['Linear\n(No Activation)', 'RBF\n(RBF Activation)']
maes_ha = [linear_mae, rbf_mae]
maes_kc = [v * HARTREE_TO_KCAL for v in maes_ha]

for ax, vals, unit, thresh in zip(
    axes,
    [maes_ha, maes_kc],
    ['Hartree', 'kcal/mol'],
    [chem_acc, 1.0]
):
    bars = ax.bar(models, vals, color=['coral', 'steelblue'],
                  edgecolor='black', lw=1.5, alpha=0.85, width=0.5)
    ax.axhline(thresh, color='green', ls='--', lw=2,
               label=f'Chemical accuracy ({thresh:.4f} {unit})')
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height()*1.01,
                f'{val:.5f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
    ax.set_ylabel(f'MAE ({unit})', fontsize=11)
    ax.set_title(f'Test MAE in {unit}', fontsize=12)
    ax.legend(fontsize=9)
    ax.grid(axis='y', alpha=0.3)

plt.suptitle('Model Performance on Test Set', fontsize=13)
plt.tight_layout()
plt.show()


In [ ]:
# Tests
assert 'linear_mae' in dir(), "linear_mae not defined"
assert 'rbf_mae'    in dir(), "rbf_mae not defined"
assert 'linear_mse' in dir(), "linear_mse not defined"
assert 'rbf_mse'    in dir(), "rbf_mse not defined"

assert linear_mae > 0, "MAE must be positive"
assert rbf_mae > 0, "MAE must be positive"
assert rbf_mae < linear_mae, "RBF should have lower MAE than linear"

assert np.isclose(linear_rmse, np.sqrt(linear_mse), atol=1e-8), \
    "linear_rmse should equal sqrt(linear_mse)"
assert np.isclose(rbf_rmse, np.sqrt(rbf_mse), atol=1e-8), \
    "rbf_rmse should equal sqrt(rbf_mse)"

assert linear_mae < 1.0, f"Linear MAE={linear_mae:.4f} is unreasonably large"
assert rbf_mae    < 0.5, f"RBF MAE={rbf_mae:.4f} is too large -- check eval mode"

HARTREE_TO_KCAL = 627.509
print("Metrics correct!")
print(f"   Linear MAE : {linear_mae:.6f} Ha = {linear_mae*HARTREE_TO_KCAL:.2f} kcal/mol")
print(f"   RBF    MAE : {rbf_mae:.6f} Ha = {rbf_mae*HARTREE_TO_KCAL:.2f} kcal/mol")
print(f"   Improvement: {linear_mae/rbf_mae:.0f}x")
chem = rbf_mae*HARTREE_TO_KCAL
print(f"   RBF chemical accuracy (< 1 kcal/mol): {'Yes' if chem < 1 else f'Not yet ({chem:.2f} kcal/mol) -- try more epochs or a deeper network'}")


---
## Explore for Yourself! (Ungraded)

### Architecture comparison

The **case study** (`NN_RBF(1, 64, 1)`) has **one hidden layer**:

```
Input  ->  Linear(1,64)  ->  RBF(64)  ->  Linear(64,1)  ->  Output
                              258 trainable parameters
```

The **explore model** (`NN_RBF(1, 32, 16, 1)`) has **two hidden layers**:

```
Input  ->  Linear(1,32)  ->  RBF(32)  ->  Linear(32,16)  ->  RBF(16)  ->  Linear(16,1)  ->  Output
                              659 trainable parameters
```

The second RBF layer lets the network combine its first-layer Gaussian detectors
non-linearly — instead of just a weighted sum of bumps, it can represent
interactions between different regions of the curve.

**Result:** 1-layer RBF: MAE ~0.027 Ha (~17 kcal/mol).  2-layer RBF: MAE ~0.002 Ha (~1.25 kcal/mol). 13x improvement.

**Things to try below:** `HIDDEN_SIZE = 64` with `USE_DEEPER = True`, or `HIDDEN_SIZE = 16`, or `HIDDEN_SIZE = 128`.


In [ ]:
# Sandbox: try different architectures
# The parameters below produced MAE=0.0020 Ha (vs 0.027 Ha for the flat 1-layer RBF).
# Change them and rerun to explore further.

# 👉 CHANGE THESE:
HIDDEN_SIZE  = 32      # try: 16, 32, 64, 128
USE_DEEPER   = True    # True = two hidden layers (1 -> H -> H//2 -> 1)
EPOCH_NUM    = 10000   # train for the same budget as the main models

# (do not change below)
torch.manual_seed(1)
if USE_DEEPER:
    explore_model = NN_RBF(1, HIDDEN_SIZE, HIDDEN_SIZE // 2, 1)
    arch_str = f'RBF  1 -> {HIDDEN_SIZE} -> {HIDDEN_SIZE//2} -> 1'
else:
    explore_model = NN_RBF(1, HIDDEN_SIZE, 1)
    arch_str = f'RBF  1 -> {HIDDEN_SIZE} -> 1'

print(f'Architecture: {arch_str}')
print(f'Parameters:   {sum(p.numel() for p in explore_model.parameters())}')

explore_model, explore_train_loss, explore_val_loss = train(
    explore_model, train_ds, x_val, y_val,
    batch_size=len(train_idx_final),
    epoch_num=EPOCH_NUM,
    print_every=EPOCH_NUM // 4
)

# Evaluate on test set
explore_model.eval()
with torch.no_grad():
    x_all_t = torch.from_numpy(X_all.reshape(-1, 1))
    explore_pred = explore_model(x_all_t).numpy()
    xt = torch.from_numpy(X_test.reshape(-1, 1))
    yt = torch.from_numpy(Y_test.reshape(-1, 1))
    exp_mae = torch.mean(torch.abs(explore_model(xt) - yt)).item()

HARTREE_TO_KCAL = 627.509
print()
print(f'Architecture  : {arch_str}')
print(f'Explore MAE   : {exp_mae:.6f} Ha = {exp_mae*HARTREE_TO_KCAL:.2f} kcal/mol')
print(f'Original RBF  : {rbf_mae:.6f} Ha = {rbf_mae*HARTREE_TO_KCAL:.2f} kcal/mol')
improvement = rbf_mae / exp_mae if exp_mae > 0 else float('inf')
print(f'Improvement   : {improvement:.1f}x better than original RBF')
chem_acc = 1.0 / HARTREE_TO_KCAL
print(f'Chemical accuracy (< 1 kcal/mol): {"YES" if exp_mae < chem_acc else f"not yet ({exp_mae*HARTREE_TO_KCAL:.2f} kcal/mol)"}')

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(X_all, Y_all,        'b-',  lw=2.5, label='FCI Reference',                          alpha=0.7)
ax.plot(X_all, rbf_pred,     'r--', lw=2,   label=f'Original RBF  (MAE={rbf_mae:.4f} Ha)')
ax.plot(X_all, explore_pred, 'g-',  lw=2,   label=f'Explore model (MAE={exp_mae:.4f} Ha)')
ax.set_xlabel(r'R ($\AA$)', fontsize=11)
ax.set_ylabel('Energy (Ha)', fontsize=11)
ax.set_title(f'Exploration: {arch_str}', fontsize=12)
ax.legend(fontsize=10)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()


---
# Reflection Questions

1. **Why does the RBF model perform so much better than the linear model?**  
   Think about the shape of the H₂ potential energy curve. What kinds of mathematical
   functions can a linear model represent? What can RBF represent that linear cannot?

2. **In `training_step`, why do we call `optimizer.zero_grad()` before `loss.backward()`?**  
   What would happen if we forgot this step?

3. **Looking at the loss curves from Exercise 7.1 — do either of the models show overfitting?**  
   How can you tell? (Hint: compare the gap between training loss and validation loss.)

4. **The RBF centres c and widths a are *learned* during training. What do you think they
   converge to after training? What physical range of R does each "neuron" specialise in?**

5. **Chemical accuracy is 1 kcal/mol. Did the RBF model achieve it?**  
   If not, what would you try to improve it — more data, more epochs, deeper network, or
   a different activation function?

---
#  Key Takeaways

- **Activation functions are essential** for fitting non-linear data. Without them, a deep
  network is mathematically identical to a single linear transformation.
- **RBF activations** act like local "detectors" — each neuron learns to respond to one
  region of the input space. Very natural for fitting smooth physical curves.
- **The training loop** has four steps: forward → loss → zero_grad → backward → step.
  Getting the order right (especially `zero_grad`) matters.
- **Validation loss** is the honest measure of how well your model generalises. Training
  loss tells you if the model is learning; validation loss tells you if it's useful.
- **Chemical accuracy (< 1 kcal/mol)** is the practical standard for computational chemistry.
  Modern neural network potentials routinely achieve this on much more complex systems.

---

